In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# ============================================================================
# CONFIGURATION — toggle between standard and NoObs results
# ============================================================================
USE_NOOBS = True  # ← Set True for no-obstacle state-only runs

BASE_DIR = Path('/home/maurits-heemskerk/Documents/Uni/Master_Thesis')
results_dir = (BASE_DIR / 'dreamer_results_local_noobs') if USE_NOOBS else (BASE_DIR / 'dreamer_results_local')

available_runs = sorted([d for d in results_dir.iterdir() if d.is_dir()])

print(f"{'[NoObs dataset]' if USE_NOOBS else '[Standard dataset]'} — {results_dir.name}")
print("Available training runs:")
for i, run in enumerate(available_runs):
    print(f"  [{i}] {run.name}")

print(f"\n✓ Loaded {len(available_runs)} available runs")


In [ ]:
import yaml

# Load ALL config files at once
all_configs = {}
for idx, run_path in enumerate(available_runs):
    config_file = run_path / 'config.yaml'
    if config_file.exists():
        with open(config_file, 'r') as f:
            all_configs[idx] = {
                'path': run_path,
                'label': run_path.name,
                'config': yaml.safe_load(f)
            }

print(f"✓ Loaded {len(all_configs)} config files")

# Helper to extract nested values
def get_nested(d, key):
    """Get value from nested dict using dot notation (e.g., 'run.train_ratio')"""
    keys = key.split('.')
    val = d
    for k in keys:
        if isinstance(val, dict):
            val = val.get(k)
        else:
            return None
    return val

# Find ALL differences across all configs
def find_all_config_differences(configs_dict):
    """Find all parameters that differ across configs"""
    
    def recurse_all(dict_list, current_path=""):
        all_keys = set()
        for d in [c.get('config', {}) for c in dict_list]:
            if isinstance(d, dict):
                all_keys.update(d.keys())
        
        results = {}
        for key in sorted(all_keys):
            new_path = f"{current_path}.{key}" if current_path else key
            
            # Get values for this parameter across all configs
            values = {}
            all_dicts = {}
            for idx, cfg_data in configs_dict.items():
                d = cfg_data.get('config', {})
                v = d.get(key) if isinstance(d, dict) else None
                values[idx] = v
                all_dicts[idx] = d.get(key) if isinstance(d, dict) else None
            
            # Check if all are dicts (recurse)
            if all(isinstance(v, dict) for v in values.values()):
                nested = recurse_all(
                    [{**cfg_data, 'config': values[idx]} for idx, cfg_data in configs_dict.items()],
                    new_path
                )
                results.update(nested)
            else:
                # Check if there are differences
                unique_values = set(str(v) for v in values.values())
                if len(unique_values) > 1:
                    results[new_path] = values
        
        return results
    
    # Start recursion with original configs
    def full_recurse(configs_dict, current_path=""):
        all_keys = set()
        for idx, cfg_data in configs_dict.items():
            d = cfg_data.get('config', {})
            if isinstance(d, dict):
                all_keys.update(d.keys())
        
        results = {}
        for key in sorted(all_keys):
            new_path = f"{current_path}.{key}" if current_path else key
            values = {}
            next_configs = {}
            
            for idx, cfg_data in configs_dict.items():
                d = cfg_data.get('config', {})
                v = d.get(key) if isinstance(d, dict) else None
                values[idx] = v
                if isinstance(v, dict):
                    next_configs[idx] = {**cfg_data, 'config': v}
            
            # Recurse if all are dicts
            if all(isinstance(v, dict) for v in values.values()) and next_configs:
                nested = full_recurse(next_configs, new_path)
                results.update(nested)
            else:
                # Check for differences
                unique_values = set(str(v) for v in values.values())
                if len(unique_values) > 1:
                    results[new_path] = values
        
        return results
    
    return full_recurse(configs_dict)

all_differences = find_all_config_differences(all_configs)

# Display comprehensive config comparison table
print("\n" + "="*120)
print("CONFIGURATION COMPARISON - ALL RUNS")
print("="*120)

if all_differences:
    # Prepare column headers
    max_label_len = max(len(cfg['label']) for cfg in all_configs.values())
    cols_header = "Parameter".ljust(40)
    for idx in sorted(all_configs.keys()):
        cols_header += f"{all_configs[idx]['label']:<{max_label_len+2}}"
    
    print(f"\n{cols_header}")
    print("-" * 120)
    
    # Display each differing parameter
    for param, values in sorted(all_differences.items()):
        row = param.ljust(40)
        for idx in sorted(values.keys()):
            val_str = str(values[idx])[:max_label_len]
            row += val_str.ljust(max_label_len+2)
        print(row)
    
    print("="*120)
    print(f"\n✓ Found {len(all_differences)} parameters that differ across runs")
else:
    print("\n✓ All configs are identical")

# Now allow selection of two runs for detailed comparison
print("\n" + "="*120)
print("SELECT TWO RUNS FOR DETAILED COMPARISON")
print("="*120)
print("\nAvailable runs:")
for idx in sorted(all_configs.keys()):
    print(f"  [{idx}] {all_configs[idx]['label']}")

print("\n👇 MODIFY THESE TO SELECT RUNS TO COMPARE:")
print("   Set run_indices_to_plot = [1, 2, 3, 4] (modify as needed)")
print("   or use: run_indices_to_plot = [run_idx_1, run_idx_2]\n")

run_indices_to_plot = [15,16,18,40,78]  # ← Modify this list to select runs

# Helper function to load metrics from a run
def load_metrics(logdir):
    """Load metrics.jsonl from a run directory"""
    metrics_file = logdir / 'metrics.jsonl'
    if metrics_file.exists():
        metrics = []
        with open(metrics_file, 'r') as f:
            for line in f:
                metrics.append(json.loads(line))
        return pd.DataFrame(metrics)
    return None

# Validate and prepare selected runs
valid_indices = [idx for idx in run_indices_to_plot if idx in all_configs]

if valid_indices:
    print(f"✓ Selected for comparison ({len(valid_indices)} runs):")
    for idx in valid_indices:
        print(f"  [{idx}] {all_configs[idx]['label']}")
else:
    print(f"⚠ Invalid indices selected")

# Build run_labels dict
run_labels = {}
for idx in all_configs.keys():
    run_labels[idx] = all_configs[idx]['label']

# Training Results Comparison

## Overview
1. **All Runs Config Diff** - Comprehensive table showing all configuration differences across all your training runs
2. **Detailed Comparison** - Select any 2+ runs to visualize and compare metrics side-by-side with different colors

In cell 2, set `run_indices_to_plot = [1, 2, 3, 4]` to compare any runs you want. All subsequent visualizations and comparisons will use this list.

In [ ]:
# HOW TO USE THIS NOTEBOOK:
# 1. Cell 2 loads all configs and displays ALL differences across all runs
# 2. In cell 2, set: run_indices_to_plot = [run_idx_1, run_idx_2]  (modify the list as needed)
#    Examples: [7, 6] or [1, 2, 3, 4] or [0, 5, 7]
# 3. The visualizations below automatically use your selection
#
# To compare more than 2 runs with different colors:
# - Cell 2: run_indices_to_plot = [1, 2, 3, 4]
# That's it! All charts and tables below will show all 4 runs.

In [ ]:
# Define all metrics to visualize
metric_info = {
    # ── World model ──────────────────────────────────────────────────────────
    "train/image_loss_mean":      ("World Model",   "Image reconstruction loss"),
    "train/rep_loss_mean":        ("World Model",   "Representation loss (encoder predictability)"),
    "train/dyn_loss_mean":        ("World Model",   "Dynamics loss (transition model)"),
    "train/model_loss_mean":      ("World Model",   "Total world-model loss"),
    "train/model_opt_loss":       ("World Model",   "Gradient-norm-scaled loss"),
    # ── RSSM health ───────────────────────────────────────────────────────────
    "train/post_ent_mean":        ("RSSM Health",   "Posterior entropy (encoder uncertainty)"),
    "train/prior_ent_mean":       ("RSSM Health",   "Prior entropy (transition model uncertainty)"),
    # ── Auxiliary heads ──────────────────────────────────────────────────────
    "train/goal_loss_mean":       ("Aux Heads",     "Goal reconstruction loss"),
    "train/position_loss_mean":   ("Aux Heads",     "Position reconstruction loss"),
    "train/orientation_loss_mean":("Aux Heads",     "Orientation reconstruction loss"),
    "train/velocity_loss_mean":   ("Aux Heads",     "Velocity reconstruction loss"),
    "train/terrain_loss_mean":    ("Aux Heads",     "Terrain reconstruction loss"),
    # ── Reward model ─────────────────────────────────────────────────────────
    "train/reward_loss_mean":     ("Reward Model",  "Reward predictor loss (NLL)"),
    "train/reward_pos_acc":       ("Reward Model",  "Positive reward accuracy (reward head, threshold>0.1)"),
    "train/reward_neg_acc":       ("Reward Model",  "Negative reward accuracy (reward head, threshold≤0.1)"),
    "train/reward_rate":          ("Reward Model",  "Fraction of positive rewards in batch"),
    "train/reward_avg":           ("Reward Model",  "Mean reward (data)"),
    "train/reward_pred":          ("Reward Model",  "Mean predicted reward (reward head)"),
    "train/reward_max_data":      ("Reward Model",  "Max |reward| in data"),
    "train/reward_max_pred":      ("Reward Model",  "Max |predicted reward| (reward head)"),
    # ── Policy & critic ──────────────────────────────────────────────────────
    "train/extr_reward_mean":     ("Env Reward",    "Mean extrinsic reward (imagined)"),
    "train/extr_reward_std":      ("Env Reward",    "Reward std dev"),
    "train/extr_return_raw_mean": ("Policy/Critic", "Mean return target (λ-return, raw)"),
    "train/extr_return_normed_mean": ("Policy/Critic", "Mean return target (normalised)"),
    "train/extr_critic_mean":     ("Policy/Critic", "Mean critic value prediction"),
    "train/adv_mean":             ("Policy/Critic", "Mean advantage signal (normed_return - base)"),
    "train/adv_std":              ("Policy/Critic", "Advantage std dev"),
    "train/extr_critic_critic_opt_loss": ("Policy/Critic", "Critic loss (×grad_scale, divide to get true NLL)"),
    "train/policy_entropy_mag":   ("Policy/Critic", "Policy entropy"),
    "train/policy_randomness_mean": ("Policy/Critic", "Policy randomness (0=deterministic, 1=uniform)"),
    # ── Training health ──────────────────────────────────────────────────────
    "train/model_opt_model_opt_grad_scale":    ("Grad Scaler", "Model gradient scale (fp16)"),
    "train/model_opt_model_opt_grad_overflow": ("Grad Scaler", "Model gradient overflow"),
    "train/extr_critic_critic_opt_critic_opt_grad_scale": ("Grad Scaler", "Critic gradient scale (fp16)"),
    "replay/size":                ("Replay",        "Replay buffer size"),
}

# Load metrics for selected runs
# Uses run_indices_to_plot defined in cell 2 above
# To compare different runs, modify cell 2: run_indices_to_plot = [1, 2, 3, 4]

# Color scheme for multiple runs
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f']
run_data = {}
run_labels_viz = {}

for i, run_idx in enumerate(run_indices_to_plot):
    if run_idx in all_configs:
        run_path = all_configs[run_idx]['path']
        run_labels_viz[run_idx] = all_configs[run_idx]['label']
        
        metrics_file = run_path / 'metrics.jsonl'
        if metrics_file.exists():
            metrics = []
            with open(metrics_file, 'r') as f:
                for line in f:
                    metrics.append(json.loads(line))
            df = pd.DataFrame(metrics)
            # Compute true critic loss: reported loss is multiplied by grad_scale for fp16
            # Divide it back out to get the actual NLL value
            scale_col = 'train/extr_critic_critic_opt_critic_opt_grad_scale'
            loss_col = 'train/extr_critic_critic_opt_loss'
            if loss_col in df.columns and scale_col in df.columns:
                df['train/extr_critic_true_loss'] = df[loss_col] / df[scale_col]
            run_data[run_idx] = df

print(f"✓ Loaded metrics for {len(run_data)} runs")

# Add derived metric to info dict
metric_info['train/extr_critic_true_loss'] = ("Policy/Critic", "Critic TRUE loss (NLL, grad-scale corrected)")

# Filter metrics that exist in at least one run
available_metrics = [col for col in metric_info.keys() 
                     if any(col in run_data[idx].columns for idx in run_data.keys())]
available_metrics.sort(key=lambda x: metric_info[x][0])

print(f"Visualizing {len(available_metrics)} metrics across {len(set(m[0] for m in metric_info.values()))} groups\n")

# Safety check: ensure we have metrics to display
if not available_metrics:
    print("⚠ WARNING: No metrics found in run data!")
    print(f"  Loaded {len(run_data)} runs, but no matching metric columns.")
    if run_data:
        print(f"\n  Available columns in first run:")
        first_run = run_data[list(run_data.keys())[0]]
        print(f"    {list(first_run.columns)[:20]}")
        print(f"    ... and {len(first_run.columns) - 20} more" if len(first_run.columns) > 20 else "")
    else:
        print(f"  No runs were loaded. Check run_indices_to_plot in cell 2.")
else:
    # Create grid: 3 columns x n rows
    ncols = 3
    nrows = (len(available_metrics) + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.5*nrows))
    if nrows == 1 and ncols == 1:
        axes = np.array([axes])  # Handle single subplot case
    axes = axes.flatten()

    for idx, metric_col in enumerate(available_metrics):
        ax = axes[idx]
        group, short_desc = metric_info[metric_col]
        
        # Plot all selected runs with different colors
        for i, run_idx in enumerate(run_indices_to_plot):
            if run_idx in run_data and metric_col in run_data[run_idx].columns:
                df = run_data[run_idx]
                color = colors[i % len(colors)]
                ax.plot(df['step'], df[metric_col], linewidth=1.2, color=color, 
                       label=run_labels_viz[run_idx], alpha=0.85)
        
        # Format plot
        ax.set_title(f"{short_desc}\n({group})", fontsize=9, fontweight='bold')
        ax.set_xlabel('Step', fontsize=9)
        ax.grid(True, alpha=0.2, linestyle='--')
        ax.tick_params(axis='both', which='major', labelsize=8)
        
        # Set y-axis limits for specific metrics to zoom in on convergence
        if metric_col in ['train/image_loss_mean']:
            ax.set_ylim(0, 200)

        if metric_col in ['train/model_loss_mean']:
            ax.set_ylim(0, 1000)
        
        # Add legend only to first plot
        if idx == 0:
            ax.legend(loc='best', fontsize=9, framealpha=0.95)

    # Hide unused subplots
    for idx in range(len(available_metrics), len(axes)):
        axes[idx].set_visible(False)

    plt.tight_layout()
    plt.show()

    print(f"\n✓ Displayed all {len(available_metrics)} metrics from {len(run_indices_to_plot)} training runs")

In [ ]:

# Compare final metric values between selected runs
print("\n" + "="*120)
print(f"FINAL METRIC VALUES COMPARISON - {len(run_indices_to_plot)} RUNS")
print("="*120)

# Build header dynamically
header = f"{'Metric':<45}"
for run_idx in run_indices_to_plot:
    header += f"{run_labels[run_idx]:<25}"
print(f"\n{header}")
print("-"*120)

# Build comparison table
for metric_col in sorted(available_metrics):
    group, short_desc = metric_info[metric_col]
    row = f"{short_desc:<45}"
    
    values_list = []
    for run_idx in run_indices_to_plot:
        if run_idx in run_data and metric_col in run_data[run_idx].columns:
            val = run_data[run_idx][metric_col].iloc[-1]
            values_list.append(val)
            row += f"{val:>24.6f}"
        else:
            row += f"{'(missing)':>24}"
    
    # Add comparison marker for 2-run case
    if len(values_list) == 2:
        val_1, val_2 = values_list
        if metric_col in ['train/extr_reward_mean', 'train/reward_pos_acc', 'train/reward_neg_acc']:
            marker = " ↑" if val_2 > val_1 else " ↓"
        else:
            marker = " ✓" if val_2 < val_1 else " ✗"
        row += marker
    
    print(row)

print("="*120)
print("\nMarkers (2-run comparison only):")
print("↑ = Run 2 is better (higher reward, higher accuracy)")
print("↓ = Run 1 is better (higher reward, higher accuracy)")
print("✓ = Run 2 is better (lower loss)")
print("✗ = Run 1 is better (lower loss)")
print("="*120)

print(f"\n📊 TRAINING RUNS SELECTED")
for run_idx in run_indices_to_plot:
    print(f"   • {run_labels[run_idx]}: {len(run_data[run_idx]) if run_idx in run_data else 0} checkpoints")

In [ ]:
# All training runs overview - comprehensive metrics
import pandas as pd

print("\n" + "="*200)
print("ALL TRAINING RUNS - COMPREHENSIVE METRICS OVERVIEW")
print("="*200)

all_runs_data = []

# Metrics to extract for each run
metrics_to_track = {
    'train/extr_reward_mean': 'Final Reward',
    'train/model_loss_mean': 'Model Loss',
    'train/image_loss_mean': 'Image Loss',
    'train/rep_loss_mean': 'Rep Loss',
    'train/dyn_loss_mean': 'Dyn Loss',
    'train/reward_loss_mean': 'Reward Loss',
    'train/reward_pos_acc': 'Reward Pos Acc',
    'train/reward_neg_acc': 'Reward Neg Acc',
    'train/velocity_loss_mean': 'Velocity Loss',
    'train/position_loss_mean': 'Position Loss',
    'train/goal_loss_mean': 'Goal Loss',
    'train/terrain_loss_mean': 'Terrain Loss',
    'train/policy_entropy_mag': 'Policy Entropy',
    'train/model_opt_model_opt_grad_scale': 'Grad Scale',
}

for run_idx, run_path in enumerate(available_runs):
    label = run_path.name  # Extract label
    
    # Load metrics
    df = load_metrics(run_path)
    
    if df is not None:
        summary_data = {'label': label, 'run_idx': run_idx, 'checkpoints': len(df)}
        
        # Extract all metrics (final values)
        for metric_key, metric_name in metrics_to_track.items():
            if metric_key in df.columns:
                summary_data[metric_name] = df[metric_key].iloc[-1]
            else:
                summary_data[metric_name] = np.nan
        
        all_runs_data.append(summary_data)

# Convert to DataFrame for nice display
if all_runs_data:
    df_summary = pd.DataFrame(all_runs_data)
    
    # Display full table
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', None)
    
    # Create display dataframe with selected columns
    display_cols = ['label', 'checkpoints'] + list(metrics_to_track.values())
    df_display = df_summary[display_cols].copy()
    df_display = df_display.round(4)
    
    print("\n" + df_display.to_string(index=False))
    print("\n" + "="*200)
    
    # Top performers rankings
    print("\n🏆 TOP PERFORMERS BY METRIC:\n")
    
    print("  Best Final Reward:")
    top_reward = df_summary.nlargest(3, 'Final Reward')[['label', 'Final Reward']]
    for i, (idx, row) in enumerate(top_reward.iterrows(), 1):
        print(f"    {i}. {row['label']:<50} → {row['Final Reward']:.6f}")
    
    print("\n  Lowest Model Loss:")
    best_loss = df_summary.nsmallest(3, 'Model Loss')[['label', 'Model Loss']]
    for i, (idx, row) in enumerate(best_loss.iterrows(), 1):
        print(f"    {i}. {row['label']:<50} → {row['Model Loss']:.6f}")
    
    print("\n  Best Reward Accuracy (Pos):")
    best_reward_acc = df_summary.nlargest(3, 'Reward Pos Acc')[['label', 'Reward Pos Acc']]
    for i, (idx, row) in enumerate(best_reward_acc.iterrows(), 1):
        print(f"    {i}. {row['label']:<50} → {row['Reward Pos Acc']:.6f}")
    
    print("\n  Lowest Image Loss:")
    best_img_loss = df_summary.nsmallest(3, 'Image Loss')[['label', 'Image Loss']]
    for i, (idx, row) in enumerate(best_img_loss.iterrows(), 1):
        print(f"    {i}. {row['label']:<50} → {row['Image Loss']:.6f}")
    
    print("\n  Lowest Terrain Loss:")
    best_terrain_loss = df_summary.nsmallest(3, 'Terrain Loss')[['label', 'Terrain Loss']]
    for i, (idx, row) in enumerate(best_terrain_loss.iterrows(), 1):
        if not np.isnan(row['Terrain Loss']):
            print(f"    {i}. {row['label']:<50} → {row['Terrain Loss']:.6f}")
else:
    print("No runs to display")


In [ ]:
# ============================================================================
# PAPER-READY GRAPHS — Clean visualization for publication
# ============================================================================
# Reload metrics for selected runs (in case run_data was overwritten)
paper_run_data = {}
for i, run_idx in enumerate(run_indices_to_plot):
    if run_idx in all_configs:
        run_path = all_configs[run_idx]['path']
        metrics_file = run_path / 'metrics.jsonl'
        if metrics_file.exists():
            metrics = []
            with open(metrics_file, 'r') as f:
                for line in f:
                    metrics.append(json.loads(line))
            df = pd.DataFrame(metrics)
            # Compute true critic loss
            scale_col = 'train/extr_critic_critic_opt_critic_opt_grad_scale'
            loss_col = 'train/extr_critic_critic_opt_loss'
            if loss_col in df.columns and scale_col in df.columns:
                df['train/extr_critic_true_loss'] = df[loss_col] / df[scale_col]
            paper_run_data[run_idx] = df

print(f"✓ Loaded paper metrics for {len(paper_run_data)} selected runs")

# ============================================================================
# CUSTOM POLICY NAMES — Edit these to define your policy names
# ============================================================================
# Map each selected run index to a custom policy name and color.
default_policy_names = ["Policy v1-A", "Policy v2-A", "Policy v3-A", "Policy v4-A", "Policy v6-A", ""]
default_policy_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
custom_policy_names = {
    run_idx: default_policy_names[i] if i < len(default_policy_names) else f"Policy {i + 1}"
    for i, run_idx in enumerate(run_indices_to_plot)
}

# Consistent colors for each policy (same across all graphs)
policy_colors = {
    run_idx: default_policy_colors[i % len(default_policy_colors)]
    for i, run_idx in enumerate(run_indices_to_plot)
}

print(f"✓ Configured custom policy names:")
for run_idx in run_indices_to_plot:
    print(f"   {custom_policy_names[run_idx]} → {policy_colors[run_idx]}")

# ============================================================================
# INDIVIDUAL METRIC GRAPHS
# ============================================================================
# Define the 3x3 paper grid by semantic row.
# Row 1: reconstruction losses; row 2: dynamics/representation losses; row 3: reward accuracies.
plot_grid = [
    [
        {
            'title': 'Orientation Reconstruction Loss',
            'metrics': [('train/orientation_loss_mean', 'Orientation')],
            'ylim': (0, 0.5),
        },
        {
            'title': 'Velocity Reconstruction Loss',
            'metrics': [('train/velocity_loss_mean', 'Velocity')],
            'ylim': (0, 0.5),
        },
        {
            'title': 'Goal Reconstruction Loss',
            'metrics': [('train/goal_loss_mean', 'Goal')],
            'ylim': (0, 0.5),
        },
    ],
    [
        {
            'title': 'Dynamics Loss',
            'metrics': [('train/dyn_loss_mean', 'Dynamics')],
        },
        {
            'title': 'Representation Loss',
            'metrics': [('train/rep_loss_mean', 'Representation')],
        },
        None,
    ],
    [
        {
            'title': 'Reward Positive Accuracy',
            'metrics': [('train/reward_pos_acc', 'Positive')],
            'ylim': (0, 1),
        },
        {
            'title': 'Reward Negative Accuracy',
            'metrics': [('train/reward_neg_acc', 'Negative')],
            'ylim': (0, 1),
        },
        None,
    ],
]

available_metric_count = 0
for row in plot_grid:
    for spec in row:
        if spec is None:
            continue
        available_metric_count += sum(
            any(metric_col in paper_run_data[idx].columns for idx in paper_run_data.keys())
            for metric_col, _ in spec['metrics'])

print(f"\n✓ Creating 3x3 paper grid with {available_metric_count} available metrics")

metric_styles = ['-', '--', ':']
nrows, ncols = 3, 3
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 10.5))

for row_idx, row in enumerate(plot_grid):
    for col_idx, spec in enumerate(row):
        ax = axes[row_idx, col_idx]
        if spec is None:
            ax.set_visible(False)
            continue

        plotted = False
        for metric_idx, (metric_col, metric_name) in enumerate(spec['metrics']):
            style = metric_styles[metric_idx % len(metric_styles)]
            for run_idx in run_indices_to_plot:
                if run_idx in paper_run_data and metric_col in paper_run_data[run_idx].columns:
                    df = paper_run_data[run_idx]
                    policy_name = custom_policy_names[run_idx]
                    color = policy_colors[run_idx]
                    label = policy_name if len(spec['metrics']) == 1 else f'{policy_name} · {metric_name}'
                    ax.plot(df['step'], df[metric_col], linestyle=style, linewidth=1.8,
                            color=color, label=label, alpha=0.85)
                    plotted = True

        if not plotted:
            ax.set_visible(False)
            continue

        ax.set_title(spec['title'], fontsize=11, fontweight='bold', pad=10)
        ax.set_xlabel('Training Step', fontsize=10)
        ax.set_ylabel('Value', fontsize=10)
        ax.grid(True, alpha=0.3, linestyle=':')
        ax.tick_params(axis='both', which='major', labelsize=9)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        if 'ylim' in spec:
            ax.set_ylim(spec['ylim'])

        # Show policy legend on the first plot and metric-combo legends on grouped reconstruction plots.
        if (row_idx, col_idx) == (0, 0) or len(spec['metrics']) > 1:
            ax.legend(fontsize=8, loc='best', framealpha=0.95)

plt.tight_layout()

save_dir = BASE_DIR / 'dreamer_SPOT_implementation' / 'notebooks' / 'figures'
save_dir.mkdir(parents=True, exist_ok=True)
png_path = save_dir / 'paper_training_metrics_grid.png'
pdf_path = save_dir / 'paper_training_metrics_grid.pdf'
fig.savefig(png_path, dpi=300, bbox_inches='tight')
fig.savefig(pdf_path, bbox_inches='tight')

plt.show()

print(f"\n✓ Generated 3x3 paper grid with {available_metric_count} available metrics")
print(f"✓ Saved figure to {png_path}")
print(f"✓ Saved figure to {pdf_path}")
print(f"\n📊 Policy Names:")
for run_idx in run_indices_to_plot:
    print(f"   {custom_policy_names[run_idx]}: {run_labels_viz[run_idx]}")


In [ ]:
# ============================================================================
# FINAL VALUES SUMMARY — Compact bar chart for the paper-ready metrics
# ============================================================================
final_metric_specs = [
    ('train/orientation_loss_mean', 'Orientation'),
    ('train/velocity_loss_mean', 'Velocity'),
    ('train/goal_loss_mean', 'Goal'),
    ('train/dyn_loss_mean', 'Dynamics'),
    ('train/rep_loss_mean', 'Representation'),
    ('train/reward_pos_acc', 'Reward + Acc'),
    ('train/reward_neg_acc', 'Reward - Acc'),
]

final_rows = []
for run_idx in run_indices_to_plot:
    if run_idx not in paper_run_data:
        continue
    df = paper_run_data[run_idx].sort_values('step')
    policy_name = custom_policy_names.get(run_idx, f'Policy {run_idx}')
    for metric_col, metric_label in final_metric_specs:
        if metric_col not in df.columns:
            continue
        values = df[['step', metric_col]].dropna()
        if values.empty:
            continue
        final_rows.append({
            'policy': policy_name,
            'metric': metric_label,
            'value': float(values[metric_col].iloc[-1]),
            'step': int(values['step'].iloc[-1]),
            'run_idx': run_idx,
        })

final_values = pd.DataFrame(final_rows)
display(final_values.pivot(index='metric', columns='policy', values='value').round(4))

metrics = [label for _, label in final_metric_specs if label in set(final_values['metric'])]
x = np.arange(len(metrics))
bar_width = 0.8 / max(1, len(run_indices_to_plot))

fig, ax = plt.subplots(figsize=(10, 3.6))
for i, run_idx in enumerate(run_indices_to_plot):
    policy_name = custom_policy_names.get(run_idx, f'Policy {run_idx}')
    run_values = []
    for metric in metrics:
        match = final_values[(final_values['run_idx'] == run_idx) & (final_values['metric'] == metric)]
        run_values.append(match['value'].iloc[0] if not match.empty else np.nan)
    offset = (i - (len(run_indices_to_plot) - 1) / 2) * bar_width
    ax.bar(x + offset, run_values, width=bar_width, label=policy_name,
           color=policy_colors.get(run_idx), alpha=0.85)

ax.set_title('Final Training Metric Values', fontsize=11, fontweight='bold', pad=10)
ax.set_ylabel('Final value', fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(metrics, rotation=25, ha='right', fontsize=9)
ax.grid(True, axis='y', alpha=0.3, linestyle=':')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=8, loc='best', framealpha=0.95)

plt.tight_layout()

save_dir = BASE_DIR / 'dreamer_SPOT_implementation' / 'notebooks' / 'figures'
save_dir.mkdir(parents=True, exist_ok=True)
png_path = save_dir / 'paper_training_final_values.png'
pdf_path = save_dir / 'paper_training_final_values.pdf'
fig.savefig(png_path, dpi=300, bbox_inches='tight')
fig.savefig(pdf_path, bbox_inches='tight')

plt.show()

print(f'\nSaved final-values chart to {png_path}')
print(f'Saved final-values chart to {pdf_path}')
